# Lenormand B4-T1 — Qwen3.8 Full64 Evidence-Conditioned Risk（Fold 0）

Subtask 2 已冻结。本 notebook 专门验证 Subtask 1：

```text
Post
 ├─ 4 个 document-only risk-card margins
 ├─ Fold-safe ModernBERT token proposer + 训练折 evidence lexicon
 ├─ Qwen3.8 对原文 candidate spans 做 A/B evidence verification
 └─ 将选中的 verbatim evidence 放回 4 个 risk cards，重新判定风险
```

主结果预先固定为：

```text
Risk margin = 0.35 × document-only + 0.65 × evidence-conditioned
```

训练和推理使用完整 64 层 QLoRA、FlashAttention 2、Flash Linear Attention 与 causal-conv1d。候选召回上限低于 `0.84` 时 notebook 会在加载 27B 前停止；Gold 字符串自身对 exact-span parser 的经验上限约为 `0.91`，所以 `0.84` 已覆盖约 92% 的可达召回。

Fold 0 预计：ModernBERT proposer 约 5–15 分钟；Full64 SFT 约 4–5 小时；Risk/Evidence 评分约 3–5 小时。所有训练 checkpoint 与评分 chunks 均写入 Drive，可断点恢复。


In [ ]:
#@title 0A. 安装基础依赖
%%capture
!pip install -q -U   "transformers>=5.8.0"   "accelerate>=1.6.0"   "peft>=0.17.0"   "bitsandbytes>=0.46.0"   "sentence-transformers>=3.4.0"   "sentencepiece>=0.2.0"   "openpyxl>=3.1.0"   "scikit-learn>=1.5.0"   "scipy>=1.13.0"   "kernels"


In [ ]:
#@title 0B. 安装 Qwen3.8 混合架构 kernels
!pip install -U "flash-linear-attention[cuda]"
!pip install -U causal-conv1d --no-build-isolation

print('新 runtime 安装完成后：Runtime → Restart session；然后从第 1 格继续。')


> 新 runtime 必须重启一次，让 Qwen3.8 重新探测 FLA/causal-conv1d。重启后不要重跑 0A/0B。


In [ ]:
#@title 1. Drive、文件与 Fold-0 开关
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import dataclasses, gc, importlib, json, math, shutil, subprocess, sys, time

ROOT = Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH = ROOT / 'train.xlsx'
ARTIFACT_ROOT = ROOT / 'results' / 'B4_TASK1_Q38_FULL64_FOLD0'

B1_PATH = ROOT / 'b1_experiments.py'
INN_PATH = ROOT / 'b1_innovation_experiments.py'
B4_PATH = ROOT / 'b4p_anchor_verifier.py'
Q38_PATH = ROOT / 'qwen38_dual_task_experiments.py'
TASK1_PATH = ROOT / 'b4_task1_q38.py'
FOLD_REFERENCE = ROOT / 'results' / 'B4P_AVC_FAST3' / 'B4P_CORE_OOF.npz'

RUN_MODERNBERT_PROPOSER = True
RUN_KERNEL_SMOKE = True
RUN_Q38_FULL64_FOLD0 = True
OVERWRITE_ADAPTER = False
SMOKE_STEPS = 3
ONLINE_TASK1_REFERENCE = 0.7577

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
required = {
    B1_PATH: None,
    INN_PATH: None,
    B4_PATH: 'B4P_RUNTIME_REVISION = "2026-08-21.qwen38-full64-kernels-v4"',
    Q38_PATH: None,
    TASK1_PATH: 'TASK1_RUNTIME_REVISION = "2026-08-24.q38-full64-official-evidence-v4"',
}
stale = [path for path, marker in required.items()
         if (not path.exists()) or (marker is not None and marker not in path.read_text(encoding='utf-8'))]
if stale:
    print('请上传本次模块并覆盖 Drive：', [path.name for path in stale])
    uploaded = files.upload()
    for path in stale:
        if path.name not in uploaded:
            raise FileNotFoundError(path)
        shutil.copy2('/content/' + path.name, path)

assert TRAIN_PATH.exists(), TRAIN_PATH
assert FOLD_REFERENCE.exists(), FOLD_REFERENCE
print(subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True,
).stdout)
drive_free_gb = shutil.disk_usage(ROOT).free / 2**30
print(f'Drive free: {drive_free_gb:.1f} GB')
if drive_free_gb < 10:
    print('WARNING: 建议至少保留 10GB 给 checkpoint、token proposer 和 score chunks。')
print('Artifacts:', ARTIFACT_ROOT)


In [ ]:
#@title 2. Kernel 硬检查与 Full64 Task-1 配置
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import torch
import transformers
import b1_experiments as b1
import b1_innovation_experiments as inn
import b4p_anchor_verifier as b4
import qwen38_dual_task_experiments as q38
import b4_task1_q38 as t1
importlib.reload(b1); importlib.reload(inn); importlib.reload(b4); importlib.reload(q38); importlib.reload(t1)

assert b4.B4P_RUNTIME_REVISION == '2026-08-21.qwen38-full64-kernels-v4'
assert t1.TASK1_RUNTIME_REVISION == '2026-08-24.q38-full64-official-evidence-v4'
kernel_status = b4.qwen35_kernel_status()
print('transformers:', transformers.__version__)
print('torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('kernel status:', kernel_status)
assert kernel_status['causal_conv1d'], 'causal-conv1d 未生效；确认安装后重启过 runtime'
assert kernel_status['flash_linear_attention'], 'FLA 未生效；确认安装后重启过 runtime'
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
assert gpu_memory_gb >= 70, f'Full64 要求 A100 80GB；当前只有 {gpu_memory_gb:.1f}GB'

CFG = t1.Task1Full64Config(
    model_name='Qwen/Qwen3.8-27B',
    fold=0,
    n_splits=3,
    max_length=1536,
    context_chars=5000,
    candidate_max_chars=180,
    candidate_caps_for_audit=(64, 96, 128, 160, 192),
    validation_candidates_per_post=64,
    evidence_negatives_per_post=3,
    evidence_top_k=3,
    evidence_margin_threshold=0.0,
    evidence_length_penalty=0.002,
    conditioned_risk_blend_weight=0.65,
    evidence_conditioned_training_fraction=0.65,
    sft_epochs=1.0,
    learning_rate=1.0e-4,
    gradient_accumulation=32,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    lora_last_n_layers=None,
    lora_target_leaves=(
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
        'in_proj_qkv', 'in_proj_z', 'in_proj_a', 'in_proj_b', 'out_proj',
    ),
    gradient_checkpointing=True,
    score_batch_size=2,
    score_chunk_size=256,
    attention_implementation='flash_attention_2',
    qwen35_fa2_position_guard=True,
    require_qwen35_fast_kernels=True,
    seed=42,
)
b4.seed_everything(CFG.seed)
torch.set_float32_matmul_precision('high')
print(json.dumps(dataclasses.asdict(CFG), indent=2))


In [ ]:
#@title 3. 数据、相同 grouped folds 与 Gold evidence 审计
bundle = b1.load_training_data(ROOT, TRAIN_PATH)
reference = np.load(FOLD_REFERENCE, allow_pickle=True)
assert reference['row_ids'].astype(str).tolist() == bundle.row_ids.astype(str).tolist()
folds = reference['folds'].astype(int)
fold_hash = b4.stable_hash({'row_ids': bundle.row_ids.tolist(), 'folds': folds.tolist()})
annotations, annotation_report = inn.prepare_evidence_annotations(bundle)
train_rows = np.flatnonzero(folds != CFG.fold)
valid_rows = np.flatnonzero(folds == CFG.fold)

print('fold sizes:', np.bincount(folds).tolist(), 'hash:', fold_hash)
print(b4.validate_folds(bundle, folds))
print('evidence annotations:', annotation_report)
print('Fold 0 train/valid:', len(train_rows), len(valid_rows))


In [ ]:
#@title 4. Fold-safe ModernBERT token proposer + paired baseline（约 5–15 分钟）
if not RUN_MODERNBERT_PROPOSER:
    raise RuntimeError('本版本要求 token proposer；不要直接烧 27B heuristic-only candidates。')

proposer = t1.prepare_modernbert_proposer_fold(
    bundle, annotations, folds, CFG.fold,
    ARTIFACT_ROOT / 'MODERNBERT_PROPOSER', epochs=2,
)
print(json.dumps(proposer.metrics, indent=2, default=str))


In [ ]:
#@title 5. 训练折 evidence lexicon + 候选召回上限 Gate（不加载 27B）
fold_dir = ARTIFACT_ROOT / 'Q38_FULL64' / f'fold_{CFG.fold}'
fold_dir.mkdir(parents=True, exist_ok=True)
lexicon_path = fold_dir / 'evidence_lexicon.json'
if lexicon_path.exists():
    lexicon = t1.EvidenceLexicon.from_json(json.loads(lexicon_path.read_text()))
    assert lexicon.training_rows == train_rows.astype(int).tolist()
else:
    lexicon = t1.fit_evidence_lexicon(bundle, annotations, train_rows, CFG)
    b4.json_dump(lexicon.to_json(), lexicon_path)

candidate_audit = t1.audit_candidate_ceiling(
    bundle, valid_rows, lexicon, CFG, token_proposals=proposer,
)
display(candidate_audit)
candidate_audit.to_csv(ARTIFACT_ROOT / 'CANDIDATE_CEILING_AUDIT.csv', index=False)

eligible = candidate_audit[candidate_audit.candidate_recall_ceiling >= 0.84]
if eligible.empty:
    raise RuntimeError(
        'Candidate ceiling < 0.84：已在加载 27B 前停止。请把本表发回，不要绕过 gate。'
    )
selected_cap = int(eligible.sort_values('candidate_cap').iloc[0].candidate_cap)
CFG = dataclasses.replace(CFG, validation_candidates_per_post=selected_cap)
print('Selected on screening Fold 0 and must be frozen for Folds 1/2:', selected_cap)
(ARTIFACT_ROOT / 'TASK1_CONFIG_SELECTED.json').write_text(
    json.dumps(dataclasses.asdict(CFG), indent=2), encoding='utf-8'
)


In [ ]:
#@title 6. Joint-SFT manifest 审计与时长估计
manifest_path = fold_dir / 'risk_evidence_pair_manifest.csv'
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
else:
    manifest = t1.build_joint_manifest(bundle, annotations, train_rows, lexicon, CFG)
    manifest.to_csv(manifest_path, index=False)

display(manifest.groupby(['task', 'target']).size().rename('pairs').reset_index())
updates = math.ceil(len(manifest) / CFG.gradient_accumulation)
print({'pairs': len(manifest), 'optimizer_updates': updates,
       'rough_train_hours_from_factor_run': updates / 221 * 3.75})
print('--- risk prompt ---')
print(manifest[manifest.task.str.startswith('risk')].iloc[0].prompt[:5000])
print('--- evidence prompt ---')
print(manifest[manifest.task.str.startswith('evidence')].iloc[0].prompt[:5000])


In [ ]:
#@title 7. Full64 + FA2/FLA 3-step smoke（约 5–12 分钟）
smoke_ok = not RUN_KERNEL_SMOKE
if RUN_KERNEL_SMOKE:
    smoke_frame = manifest.iloc[:256].copy().reset_index(drop=True)
    smoke_frame['pair_id'] = [f'task1-smoke::{i:04d}' for i in range(len(smoke_frame))]
    smoke_cfg = dataclasses.replace(CFG, sft_max_steps=SMOKE_STEPS)
    smoke_adapter = b4.train_verifier_adapter(
        q38.PromptABDataset(smoke_frame), smoke_cfg.b4_config(),
        ARTIFACT_ROOT / 'KERNEL_SMOKE', overwrite=False,
    )
    train_manifest = json.loads(
        (ARTIFACT_ROOT / 'KERNEL_SMOKE' / 'train_manifest.json').read_text()
    )
    assert train_manifest['attention_implementation'] == 'flash_attention_2'
    assert train_manifest['kernel_status']['causal_conv1d']
    assert train_manifest['kernel_status']['flash_linear_attention']
    smoke_ok = True
    print({
        'smoke_ok': smoke_ok,
        'lora_target_count': train_manifest['lora_target_count'],
        'kernel_status': train_manifest['kernel_status'],
    })
else:
    print('RUN_KERNEL_SMOKE=False；只有完全相同 Full64 kernel smoke 已通过时才允许。')


In [ ]:
#@title 8. 训练/恢复 Qwen3.8 Full64 Task-1 Fold 0（约 4–5 小时）
if RUN_Q38_FULL64_FOLD0:
    if not smoke_ok:
        raise RuntimeError('Kernel smoke 未通过，拒绝启动 Full64。')
    started = time.perf_counter()
    adapter, lexicon, manifest = t1.train_fold(
        bundle, annotations, folds, CFG,
        ARTIFACT_ROOT / 'Q38_FULL64', overwrite=OVERWRITE_ADAPTER,
    )
    print('adapter:', adapter)
    print('elapsed this session:', (time.perf_counter() - started) / 3600, 'hours')
else:
    print('RUN_Q38_FULL64_FOLD0=False')


In [ ]:
#@title 9. Document → Evidence → Conditioned Risk Fold-0 评分（可分块续跑）
metrics_path = (
    ARTIFACT_ROOT / 'Q38_FULL64' / f'fold_{CFG.fold}' /
    'EVALUATION' / 'q38_task1_fold_metrics.json'
)
if metrics_path.exists():
    task1_metrics = json.loads(metrics_path.read_text())
    print('[resume]', metrics_path)
else:
    task1_metrics = t1.evaluate_fold(
        bundle, folds, adapter, lexicon, CFG,
        ARTIFACT_ROOT / 'Q38_FULL64' / f'fold_{CFG.fold}' / 'EVALUATION',
        token_proposals=proposer,
    )

gate = t1.continuation_gate(
    task1_metrics, proposer.metrics, online_reference=ONLINE_TASK1_REFERENCE,
)
display(pd.DataFrame([
    {'system': 'ModernBERT paired proposer/baseline', **proposer.metrics},
    {'system': 'Qwen3.8 Full64 evidence-conditioned', **{
        k: v for k, v in task1_metrics.items() if not isinstance(v, (dict, list))
    }},
]))
print('Risk variants:')
print(json.dumps(task1_metrics['risk_variants'], indent=2, default=str))
print('Task-1 metrics:')
print(json.dumps(task1_metrics, indent=2, default=str))
print('Continuation gate:')
print(json.dumps(gate, indent=2, default=str))

decision = {
    'version': 'B4-T1-Q38-FULL64-FOLD0',
    'runtime_revision': t1.TASK1_RUNTIME_REVISION,
    'fold_hash': fold_hash,
    'selected_candidate_cap': CFG.validation_candidates_per_post,
    'candidate_audit': candidate_audit.to_dict(orient='records'),
    'paired_baseline': proposer.metrics,
    'challenger': task1_metrics,
    'gate': gate,
}
b4.json_dump(decision, ARTIFACT_ROOT / 'B4_TASK1_FOLD0_DECISION.json')
print('Saved:', ARTIFACT_ROOT / 'B4_TASK1_FOLD0_DECISION.json')


In [ ]:
#@title 10. 打包轻量报告（不包含 adapter/checkpoints）
report_root = ARTIFACT_ROOT / 'LIGHT_REPORT'
report_root.mkdir(parents=True, exist_ok=True)
for source in [
    ARTIFACT_ROOT / 'B4_TASK1_FOLD0_DECISION.json',
    ARTIFACT_ROOT / 'CANDIDATE_CEILING_AUDIT.csv',
    ARTIFACT_ROOT / 'TASK1_CONFIG_SELECTED.json',
    metrics_path,
    ARTIFACT_ROOT / 'Q38_FULL64' / f'fold_{CFG.fold}' / 'EVALUATION' / 'validation_predictions.csv',
]:
    if source.exists():
        shutil.copy2(source, report_root / source.name)
archive = shutil.make_archive('/content/B4_TASK1_Q38_FOLD0_REPORT', 'zip', report_root)
print('Report:', archive)
# files.download(archive)


## 决策纪律

- Candidate ceiling `< 0.84`：停止，不加载 27B；修 proposer。
- Primary Risk 永远读 `fixed_margin_blend`，不能在 Fold 0 偷选三个 risk variants 中最高者。
- Fold 0 gate 通过后，冻结 candidate cap、risk blend weight、evidence top-k/threshold，再跑 Fold 1/2。
- Evidence Phrase-F1 当前使用公开规则的 official-like 实现；提交前仍需用官方 scorer 或排行榜确认。
- 线上 `0.7577` 只是 unpaired reference；同折 ModernBERT delta 才是第一层因果对照。
